In [1]:
! pip install underthesea==9.2.11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 54.0 MB/s eta 0:00:00


In [2]:
import os
import re
import string
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from underthesea import word_tokenize

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

In [3]:
!gdown --id 1Xexp-pzJoBDflgl0ODwxu2xVXRpbMaUE
!gdown --id 1MtK80We6NqZ0lhaaEG61bNY2d3UOS0iE

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1Xexp-pzJoBDflgl0ODwxu2xVXRpbMaUE
From (redirected): https://drive.google.com/uc?id=1Xexp-pzJoBDflgl0ODwxu2xVXRpbMaUE&confirm=t&uuid=fa51b2b4-713b-4576-b3f6-94c12decb875
To: /content/segmented_cache.csv
100% 147M/147M [00:01<00:00, 87.3MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1MtK80We6NqZ0lhaaEG61bNY2d3UOS0iE
From (redirected): https://drive.google.com/uc?id=1MtK80We6NqZ0lhaaEG61bNY2d3UOS0iE&confirm=t&uuid=353021ad-c7a4-4f90-8321-c5b6b120a3f3
To: /content/news_la

In [4]:
DATA_FILE = "/content/news_labeled_clean.csv"
df = pd.read_csv(DATA_FILE)
df = df.sample(n=50_000, random_state=42).reset_index(drop=True)
print(df.shape)
print(df.columns.tolist())
df.head(3)

(50000, 10)
['id', 'url', 'category', 'label', 'title', 'desc', 'text', 'source', 'author', 'crawled_at']


,id,url,category,label,title,desc,text,source,author,crawled_at
0,103003,https://tienphong.vn/bi-thu-t-u-doan-tang-qua-doi-hinh-tiep-suc-mua-thi-vinh-phuc-ninh-binh-post1451781.tpo,giới trẻ,0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...",tienphong,Xuân Tùng,2022-07-07 14:52:37.891739
1,13977,https://www.24h.com.vn/ban-tre-cuoc-song/do-khoc-do-cuoi-voi-phan-ung-cua-be-gai-khi-bai-kiem-tra-bi-cho-cung-can-nat-c64a1369635.html,bạn trẻ - cuộc sống,0,Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát,"Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...","Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...",24h.com.vn,Theo Đinh Kim (T/h) (Đời sống & Pháp luật),2022-06-17 17:24:44.587671
2,62431,https://vov.vn/xa-hoi/tin-24h/hai-hoc-sinh-duoi-nuoc-khi-tam-bien-o-quang-tri-post953629.vov,xã hội,0,Hai học sinh đuối nước khi tắm biển ở Quảng Trị,"Trước đó, cuối buổi chiều 29/6, nhóm 3 học sinh ở xã Phong Bình, huyện Gio Linh rủ nhau tắm biển ở khu vực thôn 5, xã Gio Hải, huyện Gio Linh thì bị đuối nước. Một học sinh bơi được vào bờ và tri ...","Trước đó, cuối buổi chiều 29/6, nhóm 3 học sinh ở xã Phong Bình, huyện Gio Linh rủ nhau tắm biển ở khu vực thôn 5, xã Gio Hải, huyện Gio Linh thì bị đuối nước. Một học sinh bơi được vào bờ và tri ...",vov.vn,Đình Thiệu/VOV-Miền Trung,2022-06-30 13:04:53.246926


In [5]:
df["title_raw"] = df["title"].fillna("").astype(str)
df["text_raw"] = df["text"].fillna("").astype(str)



In [6]:
df["model_input_raw"] = (
    df["title_raw"].str.strip() + " " + df["text_raw"].str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

df[["title_raw", "text_raw", "model_input_raw"]].head(2)

,title_raw,text_raw,model_input_raw
0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...","Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hìn..."
1,Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát,"Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...","Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bê..."


In [7]:
vi_stopwords = {
    "và", "là", "của", "có", "được", "trong", "một", "những", "các", "cho", "với",
    "khi", "đã", "đang", "theo", "về", "ở", "ra", "này", "đó", "thì", "lại", "từ",
    "đến", "sau", "trên", "dưới", "tại", "bị", "do", "nên", "vẫn", "rằng", "như",
    "để", "hay", "vào", "hơn", "ít", "nhiều", "rất", "cũng", "mới"
}

In [8]:
def preprocess_vietnamese_text(text):
    text = str(text).lower().strip()

    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r"[“”‘’…–—]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()


    tokens = text.split()
    tokens = [tok for tok in tokens if tok not in vi_stopwords]
    text = " ".join(tokens)

    return text

df["model_input_clean"] = df["model_input_raw"].apply(
    lambda x: preprocess_vietnamese_text(x)
)

df[["model_input_raw", "model_input_clean"]].head(3)

,model_input_raw,model_input_clean
0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hìn...",bí thư t ư đoàn tặng quà đội hình tiếp sức mùa thi vĩnh phúc ninh bình vĩnh phúc đoàn công tác anh nguyễn tường lâm bí thư t ư đoàn làm trưởng đoàn thăm tặng quà động viên đội hình thanh niên tình...
1,"Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bê...",dở khóc dở cười phản ứng bé gái bài kiểm tra chó cưng cắn nát bà mẹ yên đài tỉnh sơn đông trung quốc đây đăng tải đoạn video ghi cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng tạo hai ...
2,"Hai học sinh đuối nước khi tắm biển ở Quảng Trị Trước đó, cuối buổi chiều 29/6, nhóm 3 học sinh ở xã Phong Bình, huyện Gio Linh rủ nhau tắm biển ở khu vực thôn 5, xã Gio Hải, huyện Gio Linh thì bị...",hai học sinh đuối nước tắm biển quảng trị trước cuối buổi chiều nhóm học sinh xã phong bình huyện gio linh rủ nhau tắm biển khu vực thôn xã gio hải huyện gio linh đuối nước học sinh bơi bờ tri hô ...


In [9]:
X = df["model_input_clean"]
y = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print("Train size:", len(X_train))
print("Test size :", len(X_test))
print("Train label distribution:")
print(y_train.value_counts(normalize=True).sort_index())
print("Test label distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Train size: 40000
Test size : 10000
Train label distribution:
label
0    0.910175
1    0.089825
Name: proportion, dtype: float64
Test label distribution:
label
0    0.9102
1    0.0898
Name: proportion, dtype: float64


In [10]:
CACHE_PATH = "/content/segmented_cache.csv"

def segment_series(text_series: pd.Series) -> pd.Series:
    """Áp dụng word_tokenize cho từng dòng, trả về Series đã tách từ."""
    return text_series.apply(
        lambda text: word_tokenize(str(text), format="text")
    )

if os.path.exists(CACHE_PATH):
    print(f"Cache tồn tại → load từ {CACHE_PATH}")
    cache_df    = pd.read_csv(CACHE_PATH)
    X_train_seg = pd.Series(cache_df[cache_df["split"] == "train"]["text_seg"].values)
    X_test_seg  = pd.Series(cache_df[cache_df["split"] == "test" ]["text_seg"].values)
    print(f"Load xong: train={len(X_train_seg)}, test={len(X_test_seg)}")
else:
    print("Chưa có cache → bắt đầu tách từ...")
    X_train_seg = segment_series(X_train.reset_index(drop=True))
    X_test_seg  = segment_series(X_test.reset_index(drop=True))

    cache_df = pd.DataFrame({
        "split":    ["train"] * len(X_train_seg) + ["test"] * len(X_test_seg),
        "text_seg": X_train_seg.tolist() + X_test_seg.tolist(),
    })
    cache_df.to_csv(CACHE_PATH, index=False)
    print(f"Đã lưu cache → {CACHE_PATH}")

# Demo kiểm tra nhanh
sample = "Chàng trai 9X Quảng Trị khởi nghiệp từ nấm sò"
print(f"\nDemo: '{sample}'")
print(f"→ '{word_tokenize(sample, format='text')}'")

# Chuyển text_seg thành danh sách các token
train_tokens = [str(text).split() for text in X_train_seg]
test_tokens = [str(text).split() for text in X_test_seg]


Cache tồn tại → load từ /content/segmented_cache.csv
Load xong: train=40000, test=10000

Demo: 'Chàng trai 9X Quảng Trị khởi nghiệp từ nấm sò'
→ 'Chàng trai 9X Quảng_Trị khởi_nghiệp từ nấm sò'


## Tạo Tokenizer, chuyển sequence và Padding

In [11]:
def to_token_list(text):
    return str(text).split()

train_tokens = X_train_seg.apply(to_token_list).tolist()
test_tokens  = X_test_seg.apply(to_token_list).tolist()

print(f"Train: {len(train_tokens)} docs")
print(f"Sample doc (10 tokens): {train_tokens[0][:10]}")

# --- TẠO TOKENIZER VÀ HỌC VOCAB TỪ TRAIN ---
# CNN cần ID của từ.
vocab_counter = Counter(word for tokens in train_tokens for word in tokens)

# Giữ lại 20,000 từ phổ biến nhất để model không quá nặng, thêm token <pad> và <unk>
MAX_VOCAB = 20000
vocab_words = ['<pad>', '<unk>'] + [w for w, c in vocab_counter.most_common(MAX_VOCAB)]
word2idx = {w: i for i, w in enumerate(vocab_words)}

# Giải thích nhanh:
# - token là 1 đơn vị từ (ở đây là từ đã segment).
# - Chỉ fit trên tập train để tránh "data leakage" (model không được biết trước từ mới ở tập test).
# - Test phải dùng từ điển của train, từ nào test có mà train không có thì quy về <unk>.

# --- TEXTS_TO_SEQUENCES & PADDING ---
MAX_LEN = 100 # Cắt/bù sao cho mọi câu đều dài đúng 100 tokens

def encode_and_pad(tokens_list, word2idx, max_len):
    seqs = []
    for tokens in tokens_list:
        # Bước 3: Chuyển text thành sequence (list các số nguyên)
        seq = [word2idx.get(w, word2idx['<unk>']) for w in tokens]

        # Bước 4: Padding
        # CNN yêu cầu input cùng kích thước (shape). Thiếu thì bù số 0 (<pad>), thừa thì cắt.
        if len(seq) < max_len:
            seq = seq + [word2idx['<pad>']] * (max_len - len(seq))
        else:
            seq = seq[:max_len]
        seqs.append(seq)
    return torch.tensor(seqs, dtype=torch.long)

X_train_pad = encode_and_pad(train_tokens, word2idx, MAX_LEN)
X_test_pad = encode_and_pad(test_tokens, word2idx, MAX_LEN)

# Chuyển nhãn thành tensor
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Đóng gói vào DataLoader
batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_pad, y_train_tensor), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_pad, y_test_tensor), batch_size=batch_size, shuffle=False)

print(f"Kích thước X_train sau padding: {X_train_pad.shape}")

Train: 40000 docs
Sample doc (10 tokens): ['giúp', 'hai', 'mẹ_con', 'quên', 'giấy_tờ', 'sân_bay', 'tài_xế', 'buồn_lòng', 'vì', 'ngờ_vực']
Kích thước X_train sau padding: torch.Size([40000, 100])


## Xây dựng mô hình CNN bằng PyTorch

In [12]:
class LSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_layers=1):
        super().__init__()
        # 1. Embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=word2idx['<pad>'])

        # 2. LSTM
        # batch_first=True vì đầu vào có dạng (batch_size, seq_len)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)

        # 3. Dense (Hidden layer)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.relu = nn.ReLU()

        # 4. Dense output (Sigmoid sẽ được gộp trong BCEWithLogitsLoss)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        x = self.embedding(x)  # -> (batch_size, seq_len, embed_dim)

        # Truyền qua LSTM
        # lstm_out: chứa output của tất cả các time step
        # hn: hidden state của bước cuối cùng
        lstm_out, (hn, cn) = self.lstm(x)

        # Ta lấy hidden state của lớp LSTM cuối cùng (nơi đã tổng hợp thông tin cả câu)
        x = hn[-1]             # -> (batch_size, hidden_dim)

        # Truyền qua các lớp Dense
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)        # -> (batch_size, 1)

        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm_model = LSTMTextClassifier(vocab_size=len(word2idx)).to(device)

## Train model

In [13]:
# Chọn Optimizer và Loss
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

epochs = 5
print("Bắt đầu training mô hình LSTM...")
for epoch in range(epochs):
    lstm_model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer_lstm.zero_grad()
        outputs = lstm_model(batch_x)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer_lstm.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

Bắt đầu training mô hình LSTM...
Epoch 1/5 | Loss: 0.2425
Epoch 2/5 | Loss: 0.1921
Epoch 3/5 | Loss: 0.1778
Epoch 4/5 | Loss: 0.1368
Epoch 5/5 | Loss: 0.1348


## Đánh giá và Dự đoán thử

In [17]:
# Đánh giá trên tập test
lstm_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = lstm_model(batch_x)

        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).float()

        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

print(f"\\nAccuracy của LSTM trên tập Test: {correct/total:.4f}")

# Thử dự đoán 1-2 câu mới

def predict_new_sentences(sentences, model, word2idx, max_len, device):
    model.eval()
    for sentence in sentences:
        # Tận dụng hàm preprocess từ baseline
        clean_text = preprocess_vietnamese_text(sentence)
        tokens = word_tokenize(clean_text, format="text").split()

        # Text -> sequence -> pad
        seq = [word2idx.get(w, word2idx['<unk>']) for w in tokens]
        if len(seq) < max_len:
            seq = seq + [word2idx['<pad>']] * (max_len - len(seq))
        else:
            seq = seq[:max_len]

        tensor_input = torch.tensor([seq], dtype=torch.long).to(device)

        with torch.no_grad():
            output = model(tensor_input)
            prob = torch.sigmoid(output).item()
            pred_class = 1 if prob >= 0.5 else 0

        print(f"\\nCâu: '{sentence}'")
        print(f"-> Xác suất lớp 1: {prob:.4f}")
        print(f"-> Dự đoán: {'Lớp 1 (Kinh tế/Doanh nghiệp...)' if pred_class == 1 else 'Lớp 0 (Khác)'}")

# Chạy thử
test_samples = [
    "VN-Index tiếp tục giảm sâu, thị trường chứng khoán chìm trong sắc đỏ, nhà đầu tư bán tháo cổ phiếu.",
    "Bí thư Đoàn thanh niên trao học bổng cho học sinh nghèo vượt khó ở vùng sâu vùng xa."
]
predict_new_sentences(test_samples, lstm_model, word2idx, MAX_LEN, device)

\nAccuracy của LSTM trên tập Test: 0.9315
\nCâu: 'VN-Index tiếp tục giảm sâu, thị trường chứng khoán chìm trong sắc đỏ, nhà đầu tư bán tháo cổ phiếu.'
-> Xác suất lớp 1: 0.7465
-> Dự đoán: Lớp 1 (Kinh tế/Doanh nghiệp...)
\nCâu: 'Bí thư Đoàn thanh niên trao học bổng cho học sinh nghèo vượt khó ở vùng sâu vùng xa.'
-> Xác suất lớp 1: 0.0018
-> Dự đoán: Lớp 0 (Khác)
